# M04 · Bayes, vraisemblance et loi Beta — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m04_bayes_beta/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m04_bayes_beta/03_notebook.ipynb`, créée par `python tools/start_chapter.py M04`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M04.1 à M04.10, M04.14 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M04.18 | Densité et moments de la loi Beta (avec `math.lgamma`) · `beta_pdf, beta_moments` | 🔨 | ★ | 20 |
|  | M04.19 | Mise à jour conjuguée et loi prédictive des prochains essais · `conjugate_beta_update, posterior_predictive` | 🔨 | ★★ | 25 |
|  | M04.20 | Le MAP d'une loi Beta, cas limites compris · `map_estimate_beta` | 🔨 | ★★ | 15 |
|  | M04.21 | La fonction Beta avec `scipy.special` : `beta`, `betaln` et l'underflow | 📦 | ★ | 15 |
| B | M04.22 | Prédire : le MAP avec Beta(1, 1) après trois échecs | 🔮 | ★ | 10 |
|  | M04.23 | Le posterior se resserre : la largeur de l'intervalle en 1/√n | 🔬 | ★★ | 20 |
|  | M04.24 | Prior, vraisemblance et posterior sur une même figure | 🎨 | ★ | 20 |
| C | M04.25 | Chasse à l'erreur : évidence oubliée, mise à jour à l'envers, mode et moyenne | 🐛 | ★ | 15 |
|  | M04.26 | Un calcul stable : comparer des vraisemblances en logarithmes (`math.lgamma`) | 🛠️ | ★ | 15 |
|  | M04.27 | Défi : traduire l'avis d'un expert en une loi Beta (moyenne, écart-type, intervalle) | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M04")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : formule de Bayes à deux et trois hypothèses, vraisemblance et maximum de vraisemblance, loi Beta, conjugaison, lissage de Laplace, MAP, intervalle de crédibilité, log-sum-exp, test A/B.
- Écrire les cinq fonctions de `mymath/bayes.py` et les tester, puis utiliser `scipy.special`.
- Prédire le MAP face à des données extrêmes, mesurer le resserrement de l'a posteriori, tracer une mise à jour, chasser des erreurs, calculer en logarithmes, traduire un avis en loi Beta.

**Rappel express.** $P(H \mid O) = \frac{P(O \mid H)\,P(H)}{P(O)}$, avec $P(O) = \sum_j P(O \mid H_j)\,P(H_j)$ ; $\mathrm{Beta}(a, b)$ : densité $\frac{\theta^{a - 1}(1 - \theta)^{b - 1}}{B(a, b)}$, moyenne $\frac{a}{a + b}$, mode $\frac{a - 1}{a + b - 2}$ ($a, b > 1$), variance $\frac{ab}{(a + b)^2(a + b + 1)}$ ; $k$ succès et $n - k$ échecs : $\mathrm{Beta}(a, b) \to \mathrm{Beta}(a + k,\ b + n - k)$ ; Laplace : $\frac{k + 1}{n + 2}$ ; log-sum-exp : $\ln\sum_j e^{\ell_j} = m + \ln\sum_j e^{\ell_j - m}$, avec $m = \max_j \ell_j$.

## Partie 0 · Vérifier tes exercices papier (✏️ M04.1 à M04.10, 🔗 M04.14)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"7/9"` ;
- une liste `[-1.5, 2.5]`, `True` / `False`, ou une lettre entre guillemets, `"B"`.

Pour M04.14 f) et g), exécute d'abord la simulation de l'énoncé : sa cellule précède celle des réponses. Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(3, 4) is accepted too, as "3/4"

**Ex M04.1 — Bayes avec deux hypothèses : un capteur de fuite d'eau**

In [ ]:
# Ex M04.1: the values you found on paper
answer_M04_1a = ...  # a) a number (4 decimals)
answer_M04_1b = ...  # b) a number (3 decimals)
answer_M04_1c = ...  # c) a number (1 decimal)
answer_M04_1d = ...  # d) a number (1 decimal)
answer_M04_1e = ...  # e) a number (1 decimal)
answer_M04_1f = ...  # f) a number (3 decimals)
answer_M04_1g = ...  # g) an irreducible fraction, as text
answer_M04_1h = ...  # h) a number (3 decimals)

for letter, answer in zip("abcdefgh", [answer_M04_1a, answer_M04_1b, answer_M04_1c, answer_M04_1d, answer_M04_1e, answer_M04_1f, answer_M04_1g, answer_M04_1h]):
    wbm.check(f"M04.1{letter}", answer)

**Ex M04.2 — Trois hypothèses et un tableau de mise à jour**

In [ ]:
# Ex M04.2: the values you found on paper
answer_M04_2a = ...  # a) a number (2 decimals)
answer_M04_2b = ...  # b) a list of 3 numbers (4 decimals)
answer_M04_2c = ...  # c) a list of 3 numbers (4 decimals)
answer_M04_2d = ...  # d) a list of 3 numbers (4 decimals)
answer_M04_2e = ...  # e) a number (4 decimals)
answer_M04_2f = ...  # f) a letter, as text
answer_M04_2g = ...  # g) a number (3 decimals)
answer_M04_2h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M04_2a, answer_M04_2b, answer_M04_2c, answer_M04_2d, answer_M04_2e, answer_M04_2f, answer_M04_2g, answer_M04_2h]):
    wbm.check(f"M04.2{letter}", answer)

**Ex M04.3 — La vraisemblance d'un taux de clic**

In [ ]:
# Ex M04.3: the values you found on paper
answer_M04_3a = ...  # a) a number (2 decimals)
answer_M04_3b = ...  # b) a number (3 decimals)
answer_M04_3c = ...  # c) a number (3 decimals)
answer_M04_3d = ...  # d) a number (4 decimals)
answer_M04_3e = ...  # e) a number (3 decimals)
answer_M04_3f = ...  # f) a number (4 decimals)
answer_M04_3g = ...  # g) a number (2 decimals)
answer_M04_3h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M04_3a, answer_M04_3b, answer_M04_3c, answer_M04_3d, answer_M04_3e, answer_M04_3f, answer_M04_3g, answer_M04_3h]):
    wbm.check(f"M04.3{letter}", answer)

**Ex M04.4 — Maximiser une vraisemblance : dériver son logarithme**

In [ ]:
# Ex M04.4: the values you found on paper
answer_M04_4a = ...  # a) a number (2 decimals)
answer_M04_4b = ...  # b) a number (2 decimals)
answer_M04_4c = ...  # c) a number (3 decimals)
answer_M04_4d = ...  # d) a number (3 decimals)
answer_M04_4e = ...  # e) a number (2 decimals)
answer_M04_4f = ...  # f) a number (2 decimals)
answer_M04_4g = ...  # g) a number (2 decimals)
answer_M04_4h = ...  # h) a number (2 decimals)

for letter, answer in zip("abcdefgh", [answer_M04_4a, answer_M04_4b, answer_M04_4c, answer_M04_4d, answer_M04_4e, answer_M04_4f, answer_M04_4g, answer_M04_4h]):
    wbm.check(f"M04.4{letter}", answer)

**Ex M04.5 — La loi Beta : normalisation, moyenne, mode, variance**

In [ ]:
# Ex M04.5: the values you found on paper
answer_M04_5a = ...  # a) an irreducible fraction, as text
answer_M04_5b = ...  # b) a number (4 decimals)
answer_M04_5c = ...  # c) an irreducible fraction, as text
answer_M04_5d = ...  # d) a number (2 decimals)
answer_M04_5e = ...  # e) an irreducible fraction, as text
answer_M04_5f = ...  # f) a number (4 decimals)
answer_M04_5g = ...  # g) an integer
answer_M04_5h = ...  # h) a list of 2 integers

for letter, answer in zip("abcdefgh", [answer_M04_5a, answer_M04_5b, answer_M04_5c, answer_M04_5d, answer_M04_5e, answer_M04_5f, answer_M04_5g, answer_M04_5h]):
    wbm.check(f"M04.5{letter}", answer)

**Ex M04.6 — Conjugaison : de Beta(a, b) à Beta(a + k, b + n − k)**

In [ ]:
# Ex M04.6: the values you found on paper
answer_M04_6a = ...  # a) a list of 2 integers
answer_M04_6b = ...  # b) a number (4 decimals)
answer_M04_6c = ...  # c) a number (3 decimals)
answer_M04_6d = ...  # d) a list of 2 integers
answer_M04_6e = ...  # e) a number (4 decimals)
answer_M04_6f = ...  # f) a number (4 decimals)
answer_M04_6g = ...  # g) a number (4 decimals)
answer_M04_6h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M04_6a, answer_M04_6b, answer_M04_6c, answer_M04_6d, answer_M04_6e, answer_M04_6f, answer_M04_6g, answer_M04_6h]):
    wbm.check(f"M04.6{letter}", answer)

**Ex M04.7 — Lissage de Laplace : la prédictive avec Beta(1, 1)**

In [ ]:
# Ex M04.7: the values you found on paper
answer_M04_7a = ...  # a) an irreducible fraction, as text
answer_M04_7b = ...  # b) an irreducible fraction, as text
answer_M04_7c = ...  # c) a number (2 decimals)
answer_M04_7d = ...  # d) an irreducible fraction, as text
answer_M04_7e = ...  # e) an irreducible fraction, as text
answer_M04_7f = ...  # f) an irreducible fraction, as text

for letter, answer in zip("abcdef", [answer_M04_7a, answer_M04_7b, answer_M04_7c, answer_M04_7d, answer_M04_7e, answer_M04_7f]):
    wbm.check(f"M04.7{letter}", answer)

**Ex M04.8 — MAP, maximum de vraisemblance et moyenne a posteriori**

In [ ]:
# Ex M04.8: the values you found on paper
answer_M04_8a = ...  # a) a number (2 decimals)
answer_M04_8b = ...  # b) a number (4 decimals)
answer_M04_8c = ...  # c) a number (2 decimals)
answer_M04_8d = ...  # d) a number (2 decimals)
answer_M04_8e = ...  # e) a number (4 decimals)
answer_M04_8f = ...  # f) a number (2 decimals)
answer_M04_8g = ...  # g) a number (2 decimals)
answer_M04_8h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M04_8a, answer_M04_8b, answer_M04_8c, answer_M04_8d, answer_M04_8e, answer_M04_8f, answer_M04_8g, answer_M04_8h]):
    wbm.check(f"M04.8{letter}", answer)

**Ex M04.9 — Un intervalle de crédibilité d'une loi Beta**

In [ ]:
# Ex M04.9: the values you found on paper
answer_M04_9a = ...  # a) a list of 2 integers
answer_M04_9b = ...  # b) a number (4 decimals)
answer_M04_9c = ...  # c) a number (5 decimals)
answer_M04_9d = ...  # d) a number (4 decimals)
answer_M04_9e = ...  # e) a number (4 decimals)
answer_M04_9f = ...  # f) a number (4 decimals)
answer_M04_9g = ...  # g) a list [low, high] (4 decimals)

for letter, answer in zip("abcdefg", [answer_M04_9a, answer_M04_9b, answer_M04_9c, answer_M04_9d, answer_M04_9e, answer_M04_9f, answer_M04_9g]):
    wbm.check(f"M04.9{letter}", answer)

**Ex M04.10 — Log-probabilités : additionner au lieu de multiplier, normaliser par log-sum-exp**

In [ ]:
# Ex M04.10: the values you found on paper
answer_M04_10a = ...  # a) a number (2 decimals)
answer_M04_10b = ...  # b) a number (3 decimals)
answer_M04_10c = ...  # c) True or False
answer_M04_10d = ...  # d) a number (3 decimals)
answer_M04_10e = ...  # e) a list of 3 numbers (4 decimals)
answer_M04_10f = ...  # f) a list of 3 numbers (4 decimals)

for letter, answer in zip("abcdef", [answer_M04_10a, answer_M04_10b, answer_M04_10c, answer_M04_10d, answer_M04_10e, answer_M04_10f]):
    wbm.check(f"M04.10{letter}", answer)

**Ex M04.14 — Un test A/B : deux taux de conversion et P(θB > θA) par simulation**

In [ ]:
# Ex M04.14 f) and g): the simulation of the statement (replace the ... by the parameters of the two posteriors)
import numpy as np

a_A, b_A = ..., ...                      # the parameters of the posterior of θA (your answer to a)
a_B, b_B = ..., ...                      # the parameters of the posterior of θB
if any(value is ... for value in (a_A, b_A, a_B, b_B)):
    print("⏳ Remplis d'abord les paramètres des deux a posteriori, puis exécute à nouveau.")
else:
    rng = np.random.default_rng(14)
    theta_A = rng.beta(a_A, b_A, 100_000)    # 100 000 draws from the posterior of A, first
    theta_B = rng.beta(a_B, b_B, 100_000)    # then 100 000 draws from the posterior of B
    print(np.mean(theta_B > theta_A), np.mean(theta_B - theta_A))

In [ ]:
# Ex M04.14: the values you found on paper
answer_M04_14a = ...  # a) a list of 2 integers
answer_M04_14b = ...  # b) a number (4 decimals)
answer_M04_14c = ...  # c) a number (4 decimals)
answer_M04_14d = ...  # d) a number (4 decimals)
answer_M04_14e = ...  # e) a number (2 decimals)
answer_M04_14f = ...  # f) a number (3 decimals)
answer_M04_14g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M04_14a, answer_M04_14b, answer_M04_14c, answer_M04_14d, answer_M04_14e, answer_M04_14f, answer_M04_14g]):
    wbm.check(f"M04.14{letter}", answer)

## Partie A · Ta librairie : la loi Beta, la conjugaison, la prédictive et le MAP

Fiche §4 à §7. Les cinq fonctions de `mymath/bayes.py`, puis la fonction Beta avec `scipy.special`.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize, special, stats

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M04.18 — Densité et moments de la loi Beta (avec `math.lgamma`) · `beta_pdf, beta_moments` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer la densité d'une loi Beta, bords compris, sans dépassement de capacité, et ses deux premiers moments.  
**Prérequis :** fiche §4 et §7 · **mymath :** `bayes.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/bayes.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Le module `math` suffit ; `math.lgamma(x)` donne $\ln\Gamma(x)$ et `math.comb(n, k)` le coefficient binomial. `scipy.stats` et `scipy.special` sont interdits : les tests comparent tes résultats à eux, et un test vérifie que tu ne t'en sers pas. Les nombres renvoyés sont des `float` de Python, sauf les paramètres de `conjugate_beta_update`.

Écris, dans cet ordre (fiche §4) :
- `beta_pdf(x, a, b)` : la densité $\frac{x^{a - 1}(1 - x)^{b - 1}}{B(a, b)}$, avec $B(a, b) = \frac{\Gamma(a)\Gamma(b)}{\Gamma(a + b)}$ ; `0.0` hors de $[0, 1]$ ; aux bords 0 et 1, les valeurs que donne la docstring ; une `ValueError` si $a \le 0$ ou $b \le 0$ ;
- `beta_moments(a, b)` : le couple (moyenne, variance).

Un piège : `math.gamma(700)` dépasse le plus grand `float` (environ $1{,}8 \times 10^{308}$) et lève une `OverflowError`. Calcule donc le **logarithme** de la densité, $(a - 1)\ln x + (b - 1)\ln(1 - x) - \ln B(a, b)$ avec $\ln B(a, b) = \ln\Gamma(a) + \ln\Gamma(b) - \ln\Gamma(a + b)$ (fiche §7), et prends l'exponentielle à la fin. Une fonction auxiliaire `_check_parameters(a, b)` évite d'écrire plusieurs fois la même vérification.

La vérification calcule :
- a) `beta_pdf(0.3, 2, 6)` (4 décimales) ;
- b) `beta_pdf(1.2, 2, 6)` (4 décimales) ;
- c) `beta_pdf(0, 1, 6)` (4 décimales) ;
- d) `math.isinf(beta_pdf(1, 2, 0.6))` : la densité est-elle infinie en 1 ?
- e) `beta_pdf(0.43, 300, 400)` (3 décimales) ;
- f) `beta_moments(3, 9)` (4 décimales) ;
- g) le nom de l'erreur levée par `beta_pdf(0.3, -2, 3)` ;
- h) le nom du type de `beta_pdf(1.2, 2, 6)`, en texte.

Dans ta copie : refais a), c) et f) à la main. Pourquoi la densité de e) dépasse-t-elle 20, alors qu'une probabilité ne dépasse jamais 1 ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M04")  # reload your saved file

In [ ]:
with checked("M04.18a"):
    wbm.check("M04.18a", mymath.bayes.beta_pdf(0.3, 2, 6), computed=True)
with checked("M04.18b"):
    wbm.check("M04.18b", mymath.bayes.beta_pdf(1.2, 2, 6), computed=True)
with checked("M04.18c"):
    wbm.check("M04.18c", mymath.bayes.beta_pdf(0, 1, 6), computed=True)
with checked("M04.18d"):
    wbm.check("M04.18d", math.isinf(mymath.bayes.beta_pdf(1, 2, 0.6)), computed=True)
with checked("M04.18e"):
    wbm.check("M04.18e", mymath.bayes.beta_pdf(0.43, 300, 400), computed=True)
with checked("M04.18f"):
    wbm.check("M04.18f", mymath.bayes.beta_moments(3, 9), computed=True)
with checked("M04.18g"):
    wbm.check("M04.18g", raised(mymath.bayes.beta_pdf, 0.3, -2, 3), computed=True)
with checked("M04.18h"):
    wbm.check("M04.18h", type(mymath.bayes.beta_pdf(1.2, 2, 6)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m04_bayes.py", "test_beta_pdf_ or test_beta_moments_ or test_bayes_uses_")

### Ex M04.19 — Mise à jour conjuguée et loi prédictive des prochains essais · `conjugate_beta_update, posterior_predictive` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la mise à jour conjuguée d'une loi Beta et la probabilité de $k$ succès dans les $n$ prochains essais, sans dépassement de capacité.  
**Prérequis :** fiche §5 ; Ex M04.18 · **mymath :** `bayes.py` · **Parcours :** C, E

Écris (fiche §5) :
- `conjugate_beta_update(a, b, successes, failures)` : les paramètres $(a + \text{succès},\ b + \text{échecs})$ de l'a posteriori ; des entiers restent des entiers ; une `ValueError` pour des paramètres invalides ou un comptage négatif ;
- `posterior_predictive(a, b, k=1, n=1)` : la probabilité de $k$ succès dans les $n$ prochains essais quand la proportion suit $\mathrm{Beta}(a, b)$, $\binom{n}{k}\frac{B(a + k,\ b + n - k)}{B(a, b)}$ ; `0.0` si $k < 0$ ou $k > n$ ; une `ValueError` si $a \le 0$, $b \le 0$ ou $n < 0$.

Attention : dans `posterior_predictive`, $k$ et $n$ comptent les essais **futurs** (notés $j$ et $m$ dans la fiche, §5), pas les données déjà vues. Comme en M04.18, additionne des logarithmes ($\ln\binom{n}{k}$ par `math.lgamma`, puis $\ln B$) et prends l'exponentielle à la fin.

La vérification calcule :
- a) `conjugate_beta_update(2, 3, 5, 10)` ;
- b) les deux paramètres renvoyés par a) sont-ils des `int` ? (`True` ou `False`)
- c) `posterior_predictive(7, 13)` (4 décimales) ;
- d) `posterior_predictive(7, 13, 2, 3)` (4 décimales) ;
- e) `posterior_predictive(7, 13, 4, 3)` (4 décimales) ;
- f) la somme des `posterior_predictive(7, 13, k, 3)` pour $k$ de 0 à 3 (4 décimales) ;
- g) le nom de l'erreur levée par `conjugate_beta_update(2, 3, -1, 4)` ;
- h) `posterior_predictive(300, 700, 12, 40)` (4 décimales).

Dans ta copie : refais c) et d) à la main, avec $\Gamma(n) = (n - 1)!$. Calcule aussi la probabilité binomiale de 2 succès en 3 essais de probabilité $\frac{7}{20}$ : pourquoi est-elle différente de d) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M04")  # reload your saved file

In [ ]:
with checked("M04.19a"):
    wbm.check("M04.19a", mymath.bayes.conjugate_beta_update(2, 3, 5, 10), computed=True)
with checked("M04.19b"):
    wbm.check("M04.19b", all(isinstance(v, int) for v in mymath.bayes.conjugate_beta_update(2, 3, 5, 10)),
              computed=True)
with checked("M04.19c"):
    wbm.check("M04.19c", mymath.bayes.posterior_predictive(7, 13), computed=True)
with checked("M04.19d"):
    wbm.check("M04.19d", mymath.bayes.posterior_predictive(7, 13, 2, 3), computed=True)
with checked("M04.19e"):
    wbm.check("M04.19e", mymath.bayes.posterior_predictive(7, 13, 4, 3), computed=True)
with checked("M04.19f"):
    wbm.check("M04.19f", sum(mymath.bayes.posterior_predictive(7, 13, k, 3) for k in range(4)), computed=True)
with checked("M04.19g"):
    wbm.check("M04.19g", raised(mymath.bayes.conjugate_beta_update, 2, 3, -1, 4), computed=True)
with checked("M04.19h"):
    wbm.check("M04.19h", mymath.bayes.posterior_predictive(300, 700, 12, 40), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m04_bayes.py", "test_conjugate_beta_update_ or test_posterior_predictive_")

### Ex M04.20 — Le MAP d'une loi Beta, cas limites compris · `map_estimate_beta` 🔨 ★★ ⏱️ 15 min
**Objectif :** programmer le mode d'une loi Beta dans tous les cas : intérieur, aux bords, ou pas de mode unique.  
**Prérequis :** fiche §6 ; Ex M04.13 · **mymath :** `bayes.py` · **Parcours :** C

Écris `map_estimate_beta(a, b)` (fiche §6) : le mode de $\mathrm{Beta}(a, b)$, c'est-à-dire le MAP d'une proportion d'a posteriori $\mathrm{Beta}(a, b)$. La formule $\frac{a - 1}{a + b - 2}$ ne vaut que si $a > 1$ et $b > 1$. Sinon, la docstring dit, cas par cas, si la densité décroît (mode 0), croît (mode 1), ou n'a pas de mode unique (une `ValueError`). L'exercice ∂ M04.13 démontre chacun de ces cas ; un croquis des densités suffit pour les retrouver.

La vérification calcule :
- a) `map_estimate_beta(7, 13)` (4 décimales) ;
- b) `map_estimate_beta(0.6, 3)` (4 décimales) ;
- c) `map_estimate_beta(3, 0.8)` (4 décimales) ;
- d) `map_estimate_beta(0.6, 1)` (4 décimales) ;
- e) le nom de l'erreur levée par `map_estimate_beta(0.8, 0.6)` ;
- f) `map_estimate_beta(12.5, 50.5)` (4 décimales) ;
- g) le nom du type de `map_estimate_beta(1, 6)`, en texte ;
- h) `map_estimate_beta(5, 17)` (4 décimales).

Dans ta copie : esquisse les densités de b), c), d) et e). h) est l'a posteriori de l'a priori uniforme après 4 succès en 20 essais : que retrouves-tu ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M04")  # reload your saved file

In [ ]:
with checked("M04.20a"):
    wbm.check("M04.20a", mymath.bayes.map_estimate_beta(7, 13), computed=True)
with checked("M04.20b"):
    wbm.check("M04.20b", mymath.bayes.map_estimate_beta(0.6, 3), computed=True)
with checked("M04.20c"):
    wbm.check("M04.20c", mymath.bayes.map_estimate_beta(3, 0.8), computed=True)
with checked("M04.20d"):
    wbm.check("M04.20d", mymath.bayes.map_estimate_beta(0.6, 1), computed=True)
with checked("M04.20e"):
    wbm.check("M04.20e", raised(mymath.bayes.map_estimate_beta, 0.8, 0.6), computed=True)
with checked("M04.20f"):
    wbm.check("M04.20f", mymath.bayes.map_estimate_beta(12.5, 50.5), computed=True)
with checked("M04.20g"):
    wbm.check("M04.20g", type(mymath.bayes.map_estimate_beta(1, 6)).__name__, computed=True)
with checked("M04.20h"):
    wbm.check("M04.20h", mymath.bayes.map_estimate_beta(5, 17), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m04_bayes.py", "test_map_estimate_beta_")

### Ex M04.21 — La fonction Beta avec `scipy.special` : `beta`, `betaln` et l'underflow 📦 ★ ⏱️ 15 min
**Objectif :** calculer la fonction Beta et son logarithme avec `scipy.special`, voir l'underflow, et calculer un rapport de fonctions Beta par les logarithmes.  
**Prérequis :** fiche §4 et §7 · **Parcours :** C

`scipy.special` regroupe les fonctions spéciales des mathématiques : `special.beta(a, b)` calcule $B(a, b)$, `special.betaln(a, b)` calcule $\ln B(a, b)$ sans passer par $B(a, b)$, et `special.gammaln(x)` calcule $\ln\Gamma(x)$ (comme `math.lgamma`, mais sur des arrays). Lis `help(special.betaln)` avant de commencer.

Calcule avec ces fonctions :
- a) $\frac{1}{B(4, 7)}$, arrondi à l'entier ;
- b) `special.beta(600, 900) == 0` : le résultat est-il nul en `float64` ? (`True` ou `False`)
- c) $\ln B(600, 900)$ (2 décimales) ;
- d) $\log_{10} B(600, 900)$ (2 décimales) ;
- e) $\frac{B(602,\ 900)}{B(600,\ 900)}$, calculé par les logarithmes (4 décimales) ;
- f) $\ln B(0{,}5 ;\ 0{,}5)$ (4 décimales).

Dans ta copie : e) est la probabilité que les deux prochains essais soient des succès quand la proportion suit $\mathrm{Beta}(600, 900)$ (fiche §5). Retrouve-la avec la formule $\frac{a(a + 1)}{(a + b)(a + b + 1)}$, puis avec ta fonction `posterior_predictive`. Que donne `special.beta(602, 900) / special.beta(600, 900)`, et pourquoi ? Que vaut $\mathrm{e}^{f}$, où $f$ est la réponse f) ?

In [ ]:
answer_M04_21a = ...  # a) an integer
answer_M04_21b = ...  # b) True or False
answer_M04_21c = ...  # c) a number (2 decimals)
answer_M04_21d = ...  # d) a number (2 decimals)
answer_M04_21e = ...  # e) a number (4 decimals)
answer_M04_21f = ...  # f) a number (4 decimals)

In [ ]:
for letter, answer in zip("abcdef", [answer_M04_21a, answer_M04_21b, answer_M04_21c, answer_M04_21d,
                                     answer_M04_21e, answer_M04_21f]):
    wbm.check(f"M04.21{letter}", answer)

## Partie B · Une prédiction, une expérience et une figure

Fiche §5 et §6. Le MAP face à des données extrêmes, le resserrement de l'a posteriori, la figure d'une mise à jour.

### Ex M04.22 — Prédire : le MAP avec Beta(1, 1) après trois échecs 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir le MAP, la moyenne et la prédictive après des observations toutes identiques, puis le vérifier.  
**Prérequis :** fiche §5 et §6 · **Parcours :** C

Un nouveau bouton « Partager » vient d'être ajouté à une application. On note $\theta$ la probabilité qu'un utilisateur qui le voit s'en serve, avec l'a priori uniforme $\mathrm{Beta}(1, 1)$. Les trois premiers utilisateurs ne s'en servent pas. Choisis, pour chaque question :
- a) le MAP de $\theta$ : (1) 0 ; (2) 0,2 ; (3) 0,25 ; (4) il n'est pas défini ;
- b) la moyenne a posteriori de $\theta$ : (1) 0 ; (2) 0,2 ; (3) 0,25 ; (4) $\frac{1}{3}$ ;
- c) le MAP si l'a priori avait été $\mathrm{Beta}(2, 2)$ : (1) 0 ; (2) 0,2 ; (3) $\frac{2}{7}$ ; (4) 0,25 ;
- d) avec l'a priori uniforme, la probabilité que les **deux** utilisateurs suivants se servent du bouton : (1) 0 ; (2) 0,04 ; (3) environ 0,067 ; (4) 0,2 ;
- e) avec l'a priori uniforme, la probabilité a posteriori que $\theta < 0{,}5$ : (1) environ 0,5 ; (2) environ 0,75 ; (3) environ 0,94 ; (4) 1.

**Avant d'exécuter quoi que ce soit**, écris tes cinq prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M04_22a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M04_22b = ...  # b) 1, 2, 3 or 4
prediction_M04_22c = ...  # c) 1, 2, 3 or 4
prediction_M04_22d = ...  # d) 1, 2, 3 or 4
prediction_M04_22e = ...  # e) 1, 2, 3 or 4

In [ ]:
for letter, prediction in zip("abcde", [prediction_M04_22a, prediction_M04_22b, prediction_M04_22c,
                                       prediction_M04_22d, prediction_M04_22e]):
    wbm.check(f"M04.22{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes cinq prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M04_22a, prediction_M04_22b, prediction_M04_22c, prediction_M04_22d, prediction_M04_22e]):
    print("⏳ Écris d'abord tes cinq prédictions, puis exécute à nouveau.")
else:
    grid_22 = np.linspace(0, 1, 100_001)
    posterior_22 = stats.beta(1 + 0, 1 + 3)                  # Beta(1, 1), then 0 success and 3 failures
    print("a) MAP (maximum of the density on a grid):", grid_22[np.argmax(posterior_22.pdf(grid_22))])
    print("b) posterior mean:", posterior_22.mean())
    print("c) MAP with the prior Beta(2, 2):", grid_22[np.argmax(stats.beta(2 + 0, 2 + 3).pdf(grid_22))])
    print("d) P(the next two trials are successes):", special.beta(1 + 2, 4) / special.beta(1, 4),
          "· the square of b):", posterior_22.mean() ** 2)
    print("e) P(θ < 0.5):", posterior_22.cdf(0.5))

**Après l'expérience**, dans ta copie : explique chaque résultat avec la loi a posteriori $\mathrm{Beta}(1, 4)$ et sa densité $4(1 - \theta)^3$. Pourquoi le MAP avec l'a priori uniforme est-il une mauvaise estimation à annoncer, et que fait l'a priori $\mathrm{Beta}(2, 2)$ ? Pourquoi d) n'est-il pas le carré de b) ?

### Ex M04.23 — Le posterior se resserre : la largeur de l'intervalle en 1/√n 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer comment la largeur d'un intervalle de crédibilité diminue avec le nombre d'essais, et comparer à $\frac{1}{\sqrt{n}}$.  
**Prérequis :** fiche §6 ; M02c §5 · **Parcours :** C

Écris `credible_width(k, n, a=1, b=1)`, qui renvoie la **largeur** de l'intervalle de crédibilité à 95 % à queues égales de la proportion, après $k$ succès en $n$ essais à partir de l'a priori $\mathrm{Beta}(a, b)$, en `float`. La méthode `ppf` d'une loi de `scipy.stats` donne ses quantiles : `stats.beta(a, b).ppf([0.025, 0.975])` renvoie les deux bornes de l'intervalle de $\mathrm{Beta}(a, b)$ (fiche §6).

La vérification appelle ta fonction pour $n = 10, 40, 160, 640, 2560$ essais, avec chaque fois 30 % de succès, et calcule :
- a) la largeur pour 3 succès en 10 essais (3 décimales) ;
- b) la largeur pour 768 succès en 2 560 essais (4 décimales) ;
- c) le rapport des largeurs pour 40 et pour 160 essais (2 décimales) ;
- d) la pente de la droite qui ajuste $\ln(\text{largeur})$ en fonction de $\ln n$ sur les cinq valeurs de $n$, par `np.polyfit` (2 décimales) ;
- e) la largeur pour 2 560 essais, multipliée par $\sqrt{2560}$ (2 décimales) ;
- f) le plus petit $n$, multiple de 10 (avec $\frac{3n}{10}$ succès), pour lequel la largeur passe sous 0,05 (un entier) ;
- g) la largeur pour 3 succès en 10 essais avec l'a priori $\mathrm{Beta}(30, 70)$ (3 décimales).

Dans ta copie : quand $n$ est multiplié par 4, par combien la largeur est-elle divisée ? Explique d) et e) avec l'écart-type d'une loi Beta, à peu près $\sqrt{\frac{p(1 - p)}{n}}$ pour $n$ grand, et la règle « $\pm 1{,}96$ écart-type » de la loi normale (M02b). Pourquoi la pente de d) n'est-elle pas exactement $-0{,}5$ ? Pourquoi g) est-il bien plus petit que a) ?

In [ ]:
def credible_width(k, n, a=1, b=1):
    """The width of the equal-tailed 95 % credible interval of a proportion, after k successes in n trials from the
    prior Beta(a, b)."""
    raise NotImplementedError("l'a posteriori Beta(a + k, b + n - k), puis ses quantiles 0.025 et 0.975")

In [ ]:
NS_23 = [10, 40, 160, 640, 2560]                     # numbers of trials, each with 30 % of successes
with checked("M04.23a"):
    wbm.check("M04.23a", credible_width(3, 10), computed=True)
with checked("M04.23b"):
    wbm.check("M04.23b", credible_width(768, 2560), computed=True)
with checked("M04.23c"):
    wbm.check("M04.23c", credible_width(12, 40) / credible_width(48, 160), computed=True)
with checked("M04.23d"):
    widths_23 = [credible_width(3 * n // 10, n) for n in NS_23]
    wbm.check("M04.23d", np.polyfit(np.log(NS_23), np.log(widths_23), 1)[0], computed=True)
with checked("M04.23e"):
    wbm.check("M04.23e", credible_width(768, 2560) * math.sqrt(2560), computed=True)
with checked("M04.23f"):
    wbm.check("M04.23f", next(n for n in range(10, 5001, 10) if credible_width(3 * n // 10, n) < 0.05),
              computed=True)
with checked("M04.23g"):
    wbm.check("M04.23g", credible_width(3, 10, 30, 70), computed=True)

### Ex M04.24 — Prior, vraisemblance et posterior sur une même figure 🎨 ★ ⏱️ 20 min
**Objectif :** tracer sur un même graphique l'a priori, la vraisemblance normalisée et l'a posteriori d'une mise à jour Beta, et y lire le compromis entre l'a priori et les données.  
**Prérequis :** fiche §5 ; R0 (matplotlib) · **Parcours :** C, E

Écris `draw_update(ax, a, b, k, n)`, qui dessine dans l'axe `ax`, sur une grille de $\theta$ d'au moins 200 points entre 0 et 1 :
- la densité de l'a priori $\mathrm{Beta}(a, b)$ ;
- la vraisemblance de $k$ succès en $n$ essais, **normalisée** pour avoir une aire de 1 : $\frac{\theta^k(1 - \theta)^{n - k}}{B(k + 1,\ n - k + 1)}$, qui est la densité de $\mathrm{Beta}(k + 1,\ n - k + 1)$ ;
- la densité de l'a posteriori $\mathrm{Beta}(a + k,\ b + n - k)$ ;

chaque courbe avec un label, puis la légende et le nom de l'axe des abscisses. Ta fonction `mymath.bayes.beta_pdf` ou `stats.beta(a, b).pdf` (qui accepte un array) donnent les densités. Si tu le veux, ajoute en pointillés verticaux les moyennes des trois lois (`ax.axvline`).

Puis trace la mise à jour d'une équipe qui pense qu'un nouveau bouton sera utilisé par 30 % des visiteurs, avec la force de 20 observations (a priori $\mathrm{Beta}(6, 14)$), et qui observe 9 utilisations sur 15 visites.

Dans ta copie : où se trouve le sommet de l'a posteriori par rapport aux deux autres ? Lequel des deux tire le plus fort, et pourquoi ? Que deviendrait la figure avec 90 utilisations sur 150 visites ?

In [ ]:
def draw_update(ax, a, b, k, n):
    """In ax: the prior Beta(a, b), the normalised likelihood of k successes in n trials, and the posterior, each
    with a label; then the legend and the name of the x axis."""
    raise NotImplementedError("une grille de θ, trois ax.plot avec label=..., ax.legend(), ax.set_xlabel(...)")

In [ ]:
with checked("M04.24 figure"):
    fig, ax = plt.subplots(figsize=(7, 4))
    try:
        created = drawn(draw_update, ax, 6, 14, 9, 15)
    except BaseException:
        plt.close(fig)                                     # no empty figure while the function is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M04.24 : draw_update ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans l'axe reçu.")
    else:
        plt.show()

**Vérifications automatiques** (sur un autre exemple ; la figure elle-même se juge à l'œil).

In [ ]:
with checked("M04.24 vérifications"):
    problems = []
    a_t, b_t, k_t, n_t = 3, 3, 2, 10                                       # another example
    expected = {"l'a priori": stats.beta(a_t, b_t), "la vraisemblance normalisée": stats.beta(k_t + 1, n_t - k_t + 1),
                "l'a posteriori": stats.beta(a_t + k_t, b_t + n_t - k_t)}
    fig, ax = plt.subplots(figsize=(6, 4))
    try:
        created = drawn(draw_update, ax, a_t, b_t, k_t, n_t)
    finally:
        plt.close(fig)                                                     # only the checks below are shown here
    if created:
        problems.append("draw_update ouvre sa propre figure : dessine dans l'axe reçu")
    else:
        curves = []
        for line in ax.get_lines():
            xs = np.asarray(line.get_xdata(), dtype=float)
            ys = np.asarray(line.get_ydata(), dtype=float)
            inside = (xs > 0.005) & (xs < 0.995)
            if inside.sum() >= 50:
                curves.append((line, xs[inside], ys[inside]))
        for name, law in expected.items():
            match = [line for line, xs, ys in curves if np.allclose(ys, law.pdf(xs), rtol=1e-6, atol=1e-6)]
            if not match:
                raw = [line for line, xs, ys in curves
                       if name == "la vraisemblance normalisée" and np.all(ys > 0)
                       and np.std(ys / (xs ** k_t * (1 - xs) ** (n_t - k_t))) < 1e-6 * np.mean(ys / (xs ** k_t * (1 - xs) ** (n_t - k_t)))]
                problems.append(f"{name} : la vraisemblance θ^k (1 − θ)^(n − k) doit être divisée par "
                                "B(k + 1, n − k + 1), pour avoir une aire de 1 comme les deux densités" if raw else
                                f"aucune courbe d'au moins 50 points sur ]0, 1[ n'est la densité de {name}"
                                + (" (a reçoit les succès, b les échecs)" if name == "l'a posteriori" else ""))
            else:
                if match[0].get_drawstyle() != "default" or match[0].get_linestyle() in ("None", "none", "", " "):
                    problems.append(f"{name} : trace une courbe continue (ax.plot), pas des points ni des marches")
                if not match[0].get_label() or match[0].get_label().startswith("_"):
                    problems.append(f"{name} : donne-lui un label, pour la légende")
        if ax.get_legend() is None:
            problems.append("il manque la légende (ax.legend())")
        if not ax.get_xlabel():
            problems.append("nomme l'axe des abscisses (θ)")
    if problems:
        print("❌ Ex M04.24 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M04.24 : l'a priori, la vraisemblance normalisée et l'a posteriori sont justes, avec une légende.")

## Partie C · Erreurs classiques, calcul stable et défi

Fiche §1, §5 à §7. Quatre erreurs de mise à jour à trouver, une log-probabilité qui ne déborde jamais, puis l'avis d'un expert à traduire en loi Beta.

### Ex M04.25 — Chasse à l'erreur : évidence oubliée, mise à jour à l'envers, mode et moyenne 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code les erreurs classiques d'une mise à jour bayésienne, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §1, §5 et §6 · **Parcours :** C

Un atelier contrôle les pièces d'un fournisseur.
- Sur le taux de défaut $\theta$ de ses pièces, l'atelier a l'a priori $\mathrm{Beta}(2, 18)$ ; sur 40 pièces contrôlées, 3 sont défectueuses (une pièce défectueuse est un « succès » pour $\theta$).
- Par ailleurs, un autre fournisseur est soit **fiable** (a priori 0,8 ; 5 % de pièces défectueuses), soit **douteux** (a priori 0,2 ; 20 % de pièces défectueuses). On tire une de ses pièces : elle est défectueuse.

Un analyste a écrit six lignes, chacune avec un commentaire qui dit ce qu'elle veut calculer. Quatre lignes sont fausses. **Sans exécuter de code**, trouve-les.

```python
b_A = 18 + 3                                   # (A) parameter b of the posterior of θ
p_B = 0.2 * 0.2                                # (B) P(douteux | défectueuse)
p_C = 0.8 * 0.05 + 0.2 * 0.2                   # (C) P(défectueuse), for the second supplier
mode_D = (2 + 3) / (2 + 18 + 40)               # (D) MAP of θ
mean_E = 5 / (5 + 55)                          # (E) posterior mean of θ
pred_F = 3 / 40                                # (F) P(the next piece is defective), for the first supplier
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des six lignes, de (A) à (F), la bonne valeur de ce que dit son commentaire (une liste de 6 nombres, 3 décimales).

Dans ta copie : pour chaque ligne fausse, quelle confusion a été faite ? Laquelle des lignes justes aurait permis de corriger (B) ?

In [ ]:
answer_M04_25a = ...  # a) letters, as text
answer_M04_25b = ...  # b) a list of 6 values (3 decimals)

In [ ]:
wbm.check("M04.25a", answer_M04_25a)
wbm.check("M04.25b", answer_M04_25b)

### Ex M04.26 — Un calcul stable : comparer des vraisemblances en logarithmes (`math.lgamma`) 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire une log-probabilité binomiale qui ne déborde jamais, cas limites compris, et s'en servir pour comparer des hypothèses sur de grands échantillons.  
**Prérequis :** fiche §7 ; R6 · **Parcours :** C

Avec 10 000 essais, la probabilité binomiale $\binom{n}{k}p^k(1 - p)^{n - k}$ ne se calcule pas directement : $p^k$ vaut 0 en `float64`, et $\binom{n}{k}$ est un entier de plus de 2 900 chiffres. La cellule fournie le montre. Les bibliothèques calculent donc son **logarithme** :

$$\ln P(K = k) = \ln\Gamma(n + 1) - \ln\Gamma(k + 1) - \ln\Gamma(n - k + 1) + k\ln p + (n - k)\ln(1 - p).$$

Écris `log_binom_pmf(k, n, p)`, qui renvoie ce logarithme en `float`, avec ces cas particuliers :
- $k < 0$ ou $k > n$ : la probabilité est nulle, renvoie `-math.inf` ;
- $p = 0$ ou $p = 1$ : `math.log(0)` lève une `ValueError`, mais $0 \times \ln 0$ vaut 0 par convention ; la probabilité vaut alors 1 ou 0 selon $k$ ;
- $p$ hors de $[0, 1]$, ou `nan` : une `ValueError` (teste `0 <= p <= 1` au début, ce qui refuse aussi `nan`).

La vérification compare ta fonction à `stats.binom.logpmf` sur neuf cas, puis une dernière cellule compare trois hypothèses sur $p$ pour 4 100 succès en 10 000 essais.

Dans ta copie : écris les trois log-vraisemblances, puis les a posteriori de la dernière cellule à la main, par log-sum-exp (fiche §7). Un écart $d$ entre deux log-vraisemblances fait un rapport $e^{d}$ entre leurs a posteriori (a priori égaux) : que donnent ici l'écart entre la première hypothèse et les deux autres, et l'écart entre les deux dernières ?

In [ ]:
# The naive computation: 0.4 ** 4100 underflows to 0.0, then a huge integer times 0.0 overflows
try:
    naive_26 = math.comb(10_000, 4100) * 0.4 ** 4100 * 0.6 ** 5900
except OverflowError as exc:
    naive_26 = f"OverflowError: {exc}"
print("naive:", naive_26)

In [ ]:
def log_binom_pmf(k, n, p):
    """The natural logarithm of the binomial probability P(K = k) for n trials of probability p; -math.inf when the
    probability is 0. ValueError if p is outside [0, 1]."""
    raise NotImplementedError("math.lgamma pour le coefficient, puis k ln p + (n - k) ln(1 - p), et les cas p = 0 ou 1")

In [ ]:
with checked("M04.26 vérifications"):
    problems = []
    cases = [(3, 10, 0.2), (0, 6, 0.0), (2, 6, 0.0), (7, 7, 1.0), (6, 7, 1.0), (4100, 10_000, 0.4),
             (12, 10, 0.3), (-1, 5, 0.5), (0, 0, 0.7)]
    for k, n, p in cases:
        expected = float(stats.binom.logpmf(k, n, p))
        try:
            got = log_binom_pmf(k, n, p)
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            problems.append(f"log_binom_pmf({k}, {n}, {p}) lève {type(exc).__name__} ({exc}) ; attendu {expected:.6g}"
                            + (" : 0 × ln 0 vaut 0 par convention, traite p = 0 et p = 1 à part" if p in (0.0, 1.0)
                               else ""))
            continue
        if not isinstance(got, float):
            problems.append(f"log_binom_pmf({k}, {n}, {p}) doit renvoyer un float de Python, pas {type(got).__name__}")
        elif math.isinf(expected):
            if got != -math.inf:
                problems.append(f"log_binom_pmf({k}, {n}, {p}) : attendu -math.inf (une probabilité nulle), "
                                f"obtenu {got!r}")
        elif not math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9):
            problems.append(f"log_binom_pmf({k}, {n}, {p}) : attendu {expected:.10g}, obtenu {got!r}")
    for p in (1.2, -0.1, float("nan")):
        try:
            log_binom_pmf(2, 5, p)
        except ValueError:
            pass
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            problems.append(f"p = {p} : lève {type(exc).__name__} ; une probabilité hors de [0, 1] demande une ValueError")
        else:
            problems.append(f"p = {p} est accepté : une probabilité hors de [0, 1] demande une ValueError")
    if problems:
        print("❌ Ex M04.26 : à revoir : " + " ; ".join(problems[:5]) + ("…" if len(problems) > 5 else "."))
    else:
        print("✅ Ex M04.26 : des log-probabilités justes, cas limites compris, même pour n = 10 000.")

**Comparer trois hypothèses.** Cette cellule utilise ta fonction (elle affiche une erreur tant que `log_binom_pmf` n'est pas écrite).

In [ ]:
with checked("M04.26 comparaison"):
    # Three hypotheses on p, equal priors: the posteriors from the log-likelihoods, by log-sum-exp
    P_26 = [0.38, 0.40, 0.42]
    log_like_26 = np.array([log_binom_pmf(4100, 10_000, p) for p in P_26])
    print("log-likelihoods:", log_like_26.round(3))
    print("posteriors (equal priors):", np.exp(log_like_26 - special.logsumexp(log_like_26)).round(4))

### Ex M04.27 — Défi : traduire l'avis d'un expert en une loi Beta (moyenne, écart-type, intervalle) 🏆 ★★★ ⏱️ 25 min
**Objectif :** construire un a priori Beta à partir d'une moyenne et d'un écart-type, puis à partir d'un intervalle, et mesurer ce qu'il pèse face aux données.  
**Prérequis :** fiche §4 à §6 ; M02a §6 (quantiles) · **Parcours :** C

Avant de lancer une nouvelle ligne de production, l'atelier interroge deux experts sur son taux de défaut $\theta$.
- Le premier dit : « environ 20 %, avec un écart-type de 5 points ».
- Le second dit : « j'ai 90 % de chances de ne pas me tromper en disant entre 10 % et 32 % », et l'on comprend que chaque queue laisse 5 %.

Traduis chaque avis en une loi Beta, puis calcule :
- a) les paramètres `[a, b]` de la loi Beta de moyenne 0,2 et d'écart-type 0,05 (1 décimale) ; pense à écrire la variance d'une loi Beta avec $m = \frac{a}{a + b}$ et $s = a + b$ : $\frac{m(1 - m)}{s + 1}$ ;
- b) les paramètres `[a, b]` de la loi Beta dont l'intervalle à 90 % à queues égales est $[0{,}10 ;\ 0{,}32]$ (1 décimale) ; aucune formule ne les donne : cherche-les par le calcul, par une recherche sur une grille ou avec `optimize.fsolve(f, x0)` (importé dans la cellule d'outils), qui part du point `x0` et cherche où la fonction `f`, qui renvoie une liste de deux nombres, vaut `[0, 0]` ;
- c) la probabilité $P(0{,}10 \le \theta \le 0{,}32)$ selon la loi de a) (4 décimales) ;
- d) la moyenne a posteriori après 7 pièces défectueuses sur 50, à partir de la loi de b), avec ses paramètres **non arrondis** (3 décimales) ;
- e) le nombre de pseudo-observations $a + b$ de la loi de b), avec ses paramètres non arrondis (1 décimale).

La vérification affiche l'intervalle et la moyenne de ta loi de b), puis vérifie tes réponses.

Dans ta copie : les deux experts sont-ils d'accord ? Lequel est le plus sûr de lui, et comment le lis-tu sur $a + b$ ? Après 50 pièces, que pèse chaque avis face aux données ?

In [ ]:
answer_M04_27a = ...  # a) [a, b] from the mean and the standard deviation (1 decimal)
answer_M04_27b = ...  # b) [a, b] whose equal-tailed 90 % interval is [0.10, 0.32] (1 decimal)
answer_M04_27c = ...  # c) P(0.10 <= θ <= 0.32) for the law of a) (4 decimals)
answer_M04_27d = ...  # d) the posterior mean after 7 defects in 50, from the law of b) (3 decimals)
answer_M04_27e = ...  # e) a + b for the law of b) (1 decimal)

In [ ]:
with checked("M04.27 cohérence"):
    if answer_M04_27b is not ... and answer_M04_27b is not None:
        a_27, b_27 = answer_M04_27b
        low_27, high_27 = stats.beta(a_27, b_27).ppf([0.05, 0.95])
        print(f"Ta loi de b) : intervalle à 90 % à queues égales [{low_27:.4f} ; {high_27:.4f}], "
              f"moyenne {a_27 / (a_27 + b_27):.4f}.")
for letter, answer in zip("abcde", [answer_M04_27a, answer_M04_27b, answer_M04_27c, answer_M04_27d, answer_M04_27e]):
    wbm.check(f"M04.27{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu appliquer la formule de Bayes à plusieurs hypothèses, une fois ou en boucle, et calculer l'évidence ?
2. Sais-tu écrire une vraisemblance, trouver son maximum, et mettre à jour une loi Beta par conjugaison (moyenne, MAP, prédictive, intervalle) ?
3. Sais-tu calculer en logarithmes (log-sum-exp, `lgamma`, `betaln`) quand les probabilités deviennent minuscules ?

**La suite** : le chapitre 4 de workbookIA (la règle de Bayes, le posterior sur une grille, la loi Beta), puis M05a, les nombres en machine et l'encadrement.